# 10 — Unified evaluation

Load the individual model prediction files, align them to the fixed 178,083-row test set, generate the combined prediction files, and recalculate the main metrics and confusion matrices. No model is trained here; run the model notebooks first.

In [3]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "data/raw/train.csv").exists()
)
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures" / "confusion_matrices"
FINAL = RESULTS / "error_analysis_full"

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

TEST = ROOT / "data/splits/full/test.csv"
EXPECTED_TEST_ROWS = 178083
MODEL_LABELS = {
    "logistic_regression": "Logistic Regression",
    "linear_svm": "Linear SVM",
    "distilbert": "DistilBERT",
    "bert_base": "BERT-base",
    "hatebert": "HateBERT",
}
PREDICTION_COLUMNS = {
    "logistic_regression": "LR_pred",
    "linear_svm": "SVM_pred",
    "distilbert": "DistilBERT_pred",
    "bert_base": "BERT_pred",
    "hatebert": "HateBERT_pred",
}


def load_runtime(experiment, model_directory):
    model_dir = RESULTS / experiment / model_directory
    for filename in ["runtime.json", "metrics.json"]:
        runtime_path = model_dir / filename
        if runtime_path.exists():
            runtime = json.loads(runtime_path.read_text(encoding="utf-8"))
            return {
                "training_runtime_seconds": runtime.get("training_seconds", np.nan)
            }
    return {"training_runtime_seconds": np.nan}


def load_predictions(path):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing single-model predictions: {path}. "
            "Run the corresponding model notebook first."
        )
    frame = pd.read_csv(path)
    pred_column = next(
        column for column in ["predicted_label", "prediction"]
        if column in frame.columns
    )
    required = {"id", pred_column}
    assert required.issubset(frame.columns)
    assert len(frame) == EXPECTED_TEST_ROWS
    assert frame["id"].is_unique
    assert set(frame[pred_column].unique()).issubset({0, 1})
    return frame[["id", pred_column]].rename(columns={pred_column: "prediction"})


def build_combined_predictions(experiment):
    canonical = pd.read_csv(TEST, usecols=["id", "comment_text", "label"])
    assert len(canonical) == EXPECTED_TEST_ROWS and canonical["id"].is_unique
    combined = canonical.rename(
        columns={"comment_text": "text", "label": "true_label"}
    )
    for model_key in PREDICTION_COLUMNS:
        path = RESULTS / experiment / model_key / "predictions.csv"
        prediction = load_predictions(path)
        assert set(prediction["id"]) == set(combined["id"])
        prediction = prediction.set_index("id").reindex(combined["id"])
        assert prediction["prediction"].notna().all()
        combined[PREDICTION_COLUMNS[model_key]] = prediction["prediction"].to_numpy(dtype="int8")
    output_names = {
        "fixed_200k": "all_model_predictions_fixed_200k.csv",
        "full_data": "all_model_predictions_full_data.csv",
    }
    output_path = FINAL / output_names[experiment]
    FINAL.mkdir(parents=True, exist_ok=True)
    combined.to_csv(output_path, index=False)
    return output_path


def evaluate(frame):
    y_true = frame["true_label"].to_numpy(dtype="int8")
    y_pred = frame["predicted_label"].to_numpy(dtype="int8")
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "toxic_f1": f1_score(y_true, y_pred, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


def save_confusion_matrix(frame, scope, model_key):
    y_true = frame["true_label"].to_numpy(dtype="int8")
    y_pred = frame["predicted_label"].to_numpy(dtype="int8")
    output_dir = FIGURES / scope
    output_dir.mkdir(parents=True, exist_ok=True)
    display = ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred,
        labels=[0, 1],
        display_labels=["non_toxic", "toxic"],
        values_format="d",
        cmap="Blues",
    )
    display.ax_.set_title(f"{MODEL_LABELS[model_key]} confusion matrix ({scope})")
    display.figure_.tight_layout()
    display.figure_.savefig(output_dir / f"{model_key}.png", dpi=180)
    plt.close(display.figure_)


def evaluate_combined_predictions(path, scope):
    frame = pd.read_csv(path)
    assert len(frame) == EXPECTED_TEST_ROWS and frame["id"].is_unique
    rows = []
    for model_key, prediction_column in PREDICTION_COLUMNS.items():
        one_model = frame[["true_label", prediction_column]].rename(
            columns={prediction_column: "predicted_label"}
        )
        save_confusion_matrix(one_model, scope, model_key)
        row = {"model": MODEL_LABELS[model_key], **evaluate(one_model)}
        row.update(load_runtime(scope, model_key))
        rows.append(row)
    return pd.DataFrame(rows)


comparison_names = {
    "fixed_200k": "model_comparison_200k",
    "full_data": "model_comparison_full_data",
}

comparison_frames = []

for experiment in ["fixed_200k", "full_data"]:
    combined_path = build_combined_predictions(experiment)
    print(experiment, "combined predictions generated:", combined_path)
    metrics = evaluate_combined_predictions(combined_path, experiment)
    metrics.insert(0, "experiment_scope", experiment)
    comparison_frames.append(metrics)
    comparison_dir = RESULTS / experiment / "comparison"
    comparison_dir.mkdir(parents=True, exist_ok=True)
    comparison_stem = comparison_names[experiment]
    metrics.to_csv(comparison_dir / f"{comparison_stem}.csv", index=False)
    display(metrics)

combined_comparison = pd.concat(comparison_frames, ignore_index=True)
combined_comparison_dir = RESULTS / "full_data" / "comparison"
combined_comparison_dir.mkdir(parents=True, exist_ok=True)
combined_comparison.to_csv(
    combined_comparison_dir / "model_comparison_combined.csv", index=False
)
print("Combined comparison generated:", combined_comparison_dir / "model_comparison_combined.csv")

fixed_200k combined predictions generated: /Users/ying/Documents/Projects/final/toxic-comment-classification/MSc_Dissertation_Supporting_Material/results/error_analysis_full/all_model_predictions_fixed_200k.csv


,experiment_scope,model,tn,fp,fn,tp,accuracy,precision,recall,toxic_f1,macro_f1
0,fixed_200k,Logistic Regression,150832,13006,3735,10510,0.905993,0.446930,0.737803,0.556659,0.752041
1,fixed_200k,Linear SVM,152765,11073,4851,9394,0.910581,0.458983,0.659459,0.541254,0.745858
2,fixed_200k,DistilBERT,160286,3552,5358,8887,0.949967,0.714446,0.623868,0.666092,0.819525
3,fixed_200k,BERT-base,159979,3859,5023,9222,0.950124,0.704992,0.647385,0.674962,0.823976
4,fixed_200k,HateBERT,159266,4572,4614,9631,0.948417,0.678096,0.676097,0.677095,0.824532


full_data combined predictions generated: /Users/ying/Documents/Projects/final/toxic-comment-classification/MSc_Dissertation_Supporting_Material/results/error_analysis_full/all_model_predictions_full_data.csv


,experiment_scope,model,tn,fp,fn,tp,accuracy,precision,recall,toxic_f1,macro_f1
0,full_data,Logistic Regression,148727,15111,2692,11553,0.900030,0.433281,0.811021,0.564815,0.754172
1,full_data,Linear SVM,148085,15753,2951,11294,0.894970,0.417569,0.792840,0.547031,0.743815
2,full_data,DistilBERT,160596,3242,5018,9227,0.953617,0.739995,0.647736,0.690799,0.832863
3,full_data,BERT-base,161002,2836,5231,9014,0.954701,0.760675,0.632783,0.690860,0.833210
4,full_data,HateBERT,160664,3174,5020,9225,0.953988,0.744012,0.647596,0.692464,0.833799


Combined comparison generated: /Users/ying/Documents/Projects/final/toxic-comment-classification/MSc_Dissertation_Supporting_Material/results/full_data/comparison/model_comparison_combined.csv
